# Lot D · Silver · étape 3 : fiche par LLM (Qwen2.5-7B-Instruct, 4 bits)

Colab : Exécution → Modifier le type d'exécution → **GPU T4**. Le test gelé n'est jamais lu.

Étape 3 : schéma « champs » + nettoyage des doublons et du suivi + **exemples mixtes** (un texte libre parmi les 3) + consignes sur les deux styles de documents.
Deux générations sur GPU : **dev** (~17 min) puis **train** (~50 min, contrôle de sur-ajustement : les règles viennent d'erreurs du dev).
Si la session se coupe : relancer les sections 1 et 2, puis la même cellule ; `--reprendre` saute les documents déjà faits (tant que l'environnement n'a pas été supprimé).

In [ ]:
!nvidia-smi -L

## 1. Récupérer la branche `lot-d-silver` (cellule ré-exécutable)

In [ ]:
%cd /content
import os
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-silver
else:
    !git clone -q -b lot-d-silver https://github.com/onsalouini/fardi-caseforge.git
%cd /content/fardi-caseforge
!git log --oneline -1

## 2. Dépendances (torch de Colab conservé)

In [ ]:
!pip install -q "transformers>=4.44" accelerate bitsandbytes scikit-learn

## 3. Essai rapide : 2 textes libres du dev
Chaque ligne doit afficher `essais 1` et `repli False`.

In [ ]:
!python generation/llm.py --split dev --graine 42 --limite 2 --out generation/sorties/essai.jsonl

## 4. Dev complet (graine 42)

In [ ]:
!python generation/llm.py --split dev --graine 42 --reprendre --out generation/sorties/llm_e3_dev_s42.jsonl

### Sauvegarde intermédiaire (au cas où la session se couperait pendant le train)

In [ ]:
!cd generation && zip -qr sorties_etape3_dev.zip sorties
from google.colab import files
files.download('generation/sorties_etape3_dev.zip')

## 5. Train complet (graine 42) : contrôle de sur-ajustement

In [ ]:
!python generation/llm.py --split train --graine 42 --reprendre --out generation/sorties/llm_e3_train_s42.jsonl

## 6. Ablation sans nettoyage (rejeu des mêmes réponses, sans GPU) et gabarit v2

In [ ]:
for split in ['dev', 'train']:
    !python generation/llm.py --split {split} --graine 42 --sans-nettoyage --rejouer generation/sorties/llm_e3_{split}_s42.journal.jsonl --out generation/sorties/llm_e3_{split}_s42_sans_nettoyage.jsonl > /dev/null
    !python generation/gabarit.py --split {split} --out generation/sorties/gabarit_v2_{split}.jsonl

## 7. Évaluation : gabarit v2, LLM étape 3, LLM sans nettoyage (dev puis train)

In [ ]:
import subprocess
def evaluer(nom, split):
    sortie = subprocess.run(['python', 'generation/evaluer.py', '--gen', f'generation/sorties/{nom}.jsonl', '--ref', f'data/final/fiches_{split}.jsonl',
                             '--docs', f'data/final/documents_{split}.jsonl', '--out', f'generation/sorties/eval_{nom}.md'],
                            capture_output=True, text=True).stdout.splitlines()
    print('=' * 15, nom)
    print(*sortie[5:16], sep=chr(10))
for split in ['dev', 'train']:
    for nom in [f'gabarit_v2_{split}', f'llm_e3_{split}_s42', f'llm_e3_{split}_s42_sans_nettoyage']:
        evaluer(nom, split)

## 8. Résumé des journaux

In [ ]:
import json
for split in ['dev', 'train']:
    j = [json.loads(l) for l in open(f'generation/sorties/llm_e3_{split}_s42.journal.jsonl', encoding='utf-8')]
    print('=' * 10, split, len(j), 'documents')
    for cle in ['retires_non_ancres', 'citations_corrigees', 'retires_sensibles', 'retires_suivi', 'retires_doublons']:
        print(f'{cle:22}:', sum(len(x.get(cle, [])) for x in j))
    print('replis gabarit        :', sum(x['repli_gabarit'] for x in j), '| essais > 1 :', sum(x['essais'] > 1 for x in j))
    print('temps moyen / doc (s) :', round(sum(x['temps_s'] for x in j) / len(j), 1))

## 9. Télécharger toutes les sorties
Dézipper dans `generation/` du dépôt local (`sorties/` est ignoré par Git).

In [ ]:
!cd generation && zip -qr sorties_etape3.zip sorties
from google.colab import files
files.download('generation/sorties_etape3.zip')